In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import requests as rq
from pathlib import Path
import re
from time import sleep
from random import random

In [2]:
data_path = Path.cwd().parent / "data"
data_path.mkdir(exist_ok=True)

In [3]:
url = "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page"
params = {}
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Referer": url,
    "Accept": "application/octet-stream" 
}

In [4]:
type_of_taxi = [
    "Yellow Taxi Trip Records",
    "Green Taxi Trip Records",
    "High Volume For-Hire Vehicle Trip Records"  # Uber
]

In [5]:
taxi_to_filename = {
    "yellow": "yellow_taxi",
    "green": "green_taxi",
    "fhvhv": "uber"
}

In [6]:
LIMIT_YEAR = 2021

In [7]:
soup = None
response = rq.get(url, params=params, headers=headers)
match response.status_code // 100:
    case 2:
        soup = BeautifulSoup(response.text, "html.parser")
    
    case _:
        print(f"[Error] Status code: {response.status_code}")

In [8]:
file_links = [tag.get("href").rstrip() for tag in soup.find_all("a", {"title": type_of_taxi})]

In [11]:
for link in file_links:
    regex_match = re.search(r"(\w+)_tripdata_(\d{4})-(\d{2})\.parquet$", link)
    year = int(regex_match.group(2))

    if year < LIMIT_YEAR:
        break

    month = int(regex_match.group(3))
    name = taxi_to_filename[regex_match.group(1)]

    session = rq.Session()
    session.headers.update(headers)

    session.get(url)
    parquet_response = session.get(link, stream=True)

    content_type = parquet_response.headers.get('Content-Type', '')

    if content_type != "binary/octet-stream":
        continue

    match parquet_response.status_code // 100:
        case 2:
            month_path = data_path / str(year) / str(month)
            month_path.mkdir(parents=True, exist_ok=True)

            with open(Path(month_path, f"{name}.parquet"), 'wb') as f:
                for chunk in parquet_response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            print("Downloaded file correctly!")
        
        case _:
            print(f"[Download Error] Status code: {response.status_code}")

    sleep(2 * (1 + random()))

Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
Downloaded file correctly!
D